# 🤖 Gauss Math Test Practice AI Agent

> **캐나다 Gauss 수학 경시대회** 기출문제를 활용하여 7~8학년 학생들의 실력 향상을 돕는 인터랙티브 **LangGraph 퀴즈 에이전트**입니다.

---

### 🎯 프로젝트 목적
캐나다 Grade 7/8 학생들이 과거 Gauss 시험 문제를 통해 실전처럼 연습할 수 있는 간단하고 직관적인 학습 환경을 제공합니다.

### 🚀 핵심 기능
*   **학년 맞춤 설정**: 학생의 수준에 맞춰 **Grade 7** 또는 **Grade 8** 중 선택 가능.
*   **기출문제 자동 출제**: Gauss 과거 문항 데이터를 기반으로 한 문제씩 순차적 출제.
*   **실시간 피드백**: 정답/오답 자동 채점 및 문제별 **핵심 해설** 제공.
*   **학습 현황 추적**: 현재 세션 내에서 풀이한 **문제 수**와 **누적 점수** 업데이트.

---

### 🏗️ 그래프 구조 (Graph Structure)
본 에이전트는 **LangGraph**를 활용하여 문제 선택, 풀이, 채점의 흐름을 상태 기반으로 관리합니다.

<img src="node-edge-diagram.jpg" width="200" height="500">

---

### 🛠️ 기술 스택 및 데이터
*   **프레임워크**: LangGraph / LangChain
*   **데이터 소스**: Gauss Math Contest Past Papers
*   **주요 로직**: 상태 기반 전이 (State-based Node & Edge)



In [8]:
from langgraph.graph import END, START, StateGraph
from typing import TypedDict, List, Optional, Literal
import subprocess
from openai import OpenAI


class Question(TypedDict):
    id: str
    text: str
    choices: List[str]   # ["A) ...", "B) ...", ...]
    correct: str         # "A" ~ "E"
    year: int
    grade: int

class GaussState(TypedDict):
    grade: int
    problem_bank: List[Question]
    current_question: Optional[Question]
    answered_count: int
    correct_count: int
    score: int
    last_answer: Optional[str]

#### Tools
def load_problems_from_past_contests(grade: int) -> List[Question]:
    """
    TODO:
      - CEMC Gauss Past Contests PDF를 파싱해서 JSON/DB로 저장해두고 여기서 로드.
      - 지금은 간단한 예시 dummy 데이터.
    """
    dummy_questions: List[Question] = [
        {
            "id": "2024-G7-Q1",
            "text": "What is 2 + 3?",
            "choices": ["A) 4", "B) 5", "C) 6", "D) 7", "E) 8"],
            "correct": "B",
            "year": 2024,
            "grade": grade,
        },
        {
            "id": "2024-G7-Q2",
            "text": "What is 10 - 4?",
            "choices": ["A) 5", "B) 6", "C) 7", "D) 8", "E) 9"],
            "correct": "B",
            "year": 2024,
            "grade": grade,
        },
    ]
    return dummy_questions




In [9]:
def init_session(state: GaussState) -> GaussState:
    """
    - 학년(grade)을 이미 state에 넣어두었다고 가정,
      실제 구현에서는 여기서 사용자에게 7/8을 물어보는 로직 추가.
    - 점수 관련 필드를 초기화.
    """
    grade = state.get("grade", 7)  # 디폴트 7학년
    return {
        "grade": grade,
        "problem_bank": [],
        "current_question": None,
        "answered_count": 0,
        "correct_count": 0,
        "score": 0,
        "last_answer": None,
    }

def load_past_contests(state: GaussState) -> GaussState:
    """
    - grade에 맞는 과거 Gauss 문제를 로딩하여 problem_bank에 저장.
    """
    grade = state["grade"]
    problems = load_problems_from_past_contests(grade)
    return {
        "problem_bank": problems
    }

def select_question(state: GaussState) -> GaussState:
    """
    - answered_count를 인덱스로 사용해서 다음 문제를 선택.
    - 더 이상 문제가 없으면 current_question을 None으로 둘 수 있음.
    """
    idx = state["answered_count"]
    problem_bank = state["problem_bank"]
    if idx < len(problem_bank):
        q = problem_bank[idx]
    else:
        q = None  # 문제 고갈 (실제 구현에서는 처리 로직 추가)
    return {
        "current_question": q
    }

def ask_question(state: GaussState) -> GaussState:
   pass

def check_answer(state: GaussState) -> GaussState:
    pass

def decide_continue(state: GaussState) -> Literal["continue", "stop"]:
    return "stop"

def end_session(state: GaussState) -> GaussState:
    pass



In [10]:
graph_builder = StateGraph(GaussState)  # Python API 기준[web:36][web:43]

# 노드 추가
graph_builder.add_node("init_session", init_session)
graph_builder.add_node("load_past_contests", load_past_contests)
graph_builder.add_node("select_question", select_question)
graph_builder.add_node("ask_question", ask_question)
graph_builder.add_node("check_answer", check_answer)
graph_builder.add_node("end_session", end_session)

# 엣지 연결
graph_builder.add_edge(START, "init_session")
graph_builder.add_edge("init_session", "load_past_contests")
graph_builder.add_edge("load_past_contests", "select_question")
graph_builder.add_edge("select_question", "ask_question")
graph_builder.add_edge("ask_question", "check_answer")

# 조건부 엣지 (라우터)
graph_builder.add_conditional_edges(
    "check_answer",
    decide_continue,           # router 함수
    {
        "continue": "select_question",
        "stop": "end_session",
    },
)

graph_builder.add_edge("end_session", END)

graph = graph_builder.compile()

#graph

graph.invoke(
    {
        "grade": 7,
        "problem_bank": [],
        "current_question": None,
        "answered_count": 0,
        "correct_count": 0,
        "score": 0,
        "last_answer": None,
    }
)

{'grade': 7,
 'problem_bank': [{'id': '2024-G7-Q1',
   'text': 'What is 2 + 3?',
   'choices': ['A) 4', 'B) 5', 'C) 6', 'D) 7', 'E) 8'],
   'correct': 'B',
   'year': 2024,
   'grade': 7},
  {'id': '2024-G7-Q2',
   'text': 'What is 10 - 4?',
   'choices': ['A) 5', 'B) 6', 'C) 7', 'D) 8', 'E) 9'],
   'correct': 'B',
   'year': 2024,
   'grade': 7}],
 'current_question': {'id': '2024-G7-Q1',
  'text': 'What is 2 + 3?',
  'choices': ['A) 4', 'B) 5', 'C) 6', 'D) 7', 'E) 8'],
  'correct': 'B',
  'year': 2024,
  'grade': 7},
 'answered_count': 0,
 'correct_count': 0,
 'score': 0,
 'last_answer': None}